# 05 Tokenization

Replace customer email addresses with irreversible random tokens (UUIDs). Creates a tokenized customer table plus a secured email-token lookup table in `pii_demo.pii_target`.

In [0]:
%python
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import uuid

# Read the source customer table
source_df = spark.table("pii_demo.pii_source.customers")

print(f"Total records: {source_df.count()}")
display(source_df.limit(5))

In [0]:
%python
# Create a UDF to generate unique tokens (UUIDs)
def generate_token():
    return str(uuid.uuid4())

generate_token_udf = F.udf(generate_token)

# Get distinct emails and generate tokens for each
email_lookup_df = source_df.select("email").distinct() \
    .withColumn("token", generate_token_udf())

# Build lookup tables with unique tokens for the other PII columns
full_name_lookup_df = source_df.select("full_name").distinct() \
    .withColumn("token", generate_token_udf())

phone_lookup_df = source_df.select("phone").distinct() \
    .withColumn("token", generate_token_udf())

dob_lookup_df = source_df.select("date_of_birth").distinct() \
    .withColumn("token", generate_token_udf())

address_lookup_df = source_df.select("address").distinct() \
    .withColumn("token", generate_token_udf())

print(f"Unique emails: {email_lookup_df.count()}")
print(f"Unique full names: {full_name_lookup_df.count()}")
print(f"Unique phone numbers: {phone_lookup_df.count()}")
print(f"Unique dates of birth: {dob_lookup_df.count()}")
print(f"Unique addresses: {address_lookup_df.count()}")
display(email_lookup_df.limit(10))
display(full_name_lookup_df.limit(10))
display(phone_lookup_df.limit(10))
display(dob_lookup_df.limit(10))
display(address_lookup_df.limit(10))

In [0]:
%python
# Join the source data with all PII lookups to replace PII values with tokens
customers_tokenized_df = source_df \
    .join(email_lookup_df.select("email", F.col("token").alias("email_token")), on="email", how="left") \
    .join(full_name_lookup_df.select("full_name", F.col("token").alias("full_name_token")), on="full_name", how="left") \
    .join(phone_lookup_df.select("phone", F.col("token").alias("phone_token")), on="phone", how="left") \
    .join(dob_lookup_df.select("date_of_birth", F.col("token").alias("dob_token")), on="date_of_birth", how="left") \
    .join(address_lookup_df.select("address", F.col("token").alias("address_token")), on="address", how="left") \
    .select(
        "customer_id",
        F.col("email_token").alias("email_token"),
        F.col("full_name_token").alias("full_name_token"),
        F.col("phone_token").alias("phone_token"),
        "ssn",
        F.col("dob_token").alias("dob_token"),
        F.col("address_token").alias("address_token"),
        "city",
        "country"
    )

print(f"Tokenized records: {customers_tokenized_df.count()}")
display(customers_tokenized_df.limit(10))

In [0]:
%python
# Write the tokenized customer data to target table (schema changed vs existing table, so allow the overwrite to replace it)
customers_tokenized_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("pii_demo.pii_target.customers_tokenized")

# Write the email token lookup table
email_lookup_df.select("token", "email").write.mode("overwrite").saveAsTable("pii_demo.pii_target.email_token_lookup")

# Write the other PII token lookup tables
full_name_lookup_df.select("token", "full_name").write.mode("overwrite").saveAsTable("pii_demo.pii_target.full_name_token_lookup")

phone_lookup_df.select("token", "phone").write.mode("overwrite").saveAsTable("pii_demo.pii_target.phone_token_lookup")

dob_lookup_df.select("token", "date_of_birth").write.mode("overwrite").saveAsTable("pii_demo.pii_target.dob_token_lookup")

address_lookup_df.select("token", "address").write.mode("overwrite").saveAsTable("pii_demo.pii_target.address_token_lookup")

print("✓ Tokenization complete!")
print("✓ Created table: pii_demo.pii_target.customers_tokenized")
print("✓ Created table: pii_demo.pii_target.email_token_lookup")
print("✓ Created table: pii_demo.pii_target.full_name_token_lookup")
print("✓ Created table: pii_demo.pii_target.phone_token_lookup")
print("✓ Created table: pii_demo.pii_target.dob_token_lookup")
print("✓ Created table: pii_demo.pii_target.address_token_lookup")

In [0]:
select * from pii_demo.pii_target.customers_tokenized

In [0]:
select * from pii_demo.pii_target.email_token_lookup